[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/langgraph/langgraph_evaluator.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# LangGraph：评估与优化循环
本教程使用 LangGraph 构建代码生成反馈循环：生成 LLM 编写代码，评估 LLM 提供结构化评审。这种迭代模式通过多步骤改进输出，直到达到预定义的成功标准。

工作流包含：

根据反馈生成或修改代码的**生成 LLM**。

给出 pass 或 fail 评级，并在必要时提供反馈的**评估 LLM**。

在评估通过之前持续调用生成器的 **LangGraph 状态机**。

为获得完整可观测性和生产级能力，图中接入了 Phoenix 追踪，可检查每次生成与评估、查看模型产物，并理解通过或失败的原因。

In [ ]:
!pip install langgraph langchain langchain_community "arize-phoenix" arize-phoenix-otel openinference-instrumentation-langchain

In [ ]:
!pip install langchain_openai

In [ ]:
import os
from getpass import getpass

from langgraph.graph import END, START, StateGraph

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

# 配置 Phoenix 追踪
请前往 https://app.phoenix.arize.com/ 生成 API 密钥，以便使用 Phoenix 追踪 LangGraph 应用。

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="Evaluator-Optimizer", auto_instrument=True)

# 评估与优化：代码编写循环
---------------------------------------
输入：problem_spec（函数或程序的自然语言描述）

输出：经过改进并被接受的代码（Python 字符串）


In [ ]:
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

LLM
----
• generator_llm：编写或重写代码

• evaluator_llm：通过结构化输出评定代码

In [ ]:
generator_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)
evaluator_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

为让评估 LLM 提供结构化、可靠的反馈，定义名为 Review 的 Pydantic 模式，确保每次评估都有二元评级（pass 或 fail），并在代码需要改进时提供反馈。

将该模式绑定到评估 LLM，能够保证输出格式一致，并简化图中的路由逻辑。这是闭合反馈回路、驱动迭代优化的关键步骤。

In [ ]:
class Review(BaseModel):
    grade: Literal["pass", "fail"] = Field(description="代码是否完整解决了问题？")
    feedback: str = Field(description="若 grade=='fail'，请给出具体、可操作的反馈。")


evaluator = evaluator_llm.with_structured_output(Review)

# LangGraph 共享状态
定义评估与优化循环的共享状态，跟踪问题描述、生成代码、反馈和评估评级。

In [ ]:
class State(TypedDict):
    problem_spec: str
    code: str
    feedback: str
    grade: str  # pass / fail

# 节点函数：生成器与评估器
这些节点驱动评估与优化循环。code_generator 使用生成 LLM，根据任务与先前反馈生成或修改 Python 代码。code_evaluator 使用结构化评估 LLM 模拟代码评审，在推理中按规格检查代码，返回二元评级，并在必要时提出建设性反馈。反馈随后传回生成器，直到代码通过。

In [ ]:
def code_generator(state: State):
    """根据反馈编写或改进代码。"""
    prompt = (
        "你是 Python 开发专家。\n"
        "请编写清晰、高效、符合 PEP‑8 的代码，解决下列任务。\n"
        "如果提供了反馈，请据此修改先前代码。\n\n"
        f"### 任务\n{state['problem_spec']}\n\n"
    )
    if state.get("feedback"):
        prompt += f"### 先前评审反馈\n{state['feedback']}\n"

    msg = generator_llm.invoke(prompt)
    return {"code": msg.content}


def code_evaluator(state: State):
    """由 LLM 评审代码方案。"""
    review = evaluator.invoke(
        [
            SystemMessage(
                content=(
                    "你是严格的代码评审者。"
                    "通过推理和模拟测试判断代码是否符合规格。"
                    "若不符合，请给出简洁、可操作的反馈。"
                )
            ),
            HumanMessage(
                content=(
                    f"### 问题\n{state['problem_spec']}\n\n### 待评审代码\n{state['code']}"
                )
            ),
        ]
    )
    return {"grade": review.grade, "feedback": review.feedback}

# 路由逻辑
定义简单路由函数 route，以支持迭代改进。评估结束后，函数检查评级：通过则结束，失败则返回生成器再次修改，使 LLM 能根据反馈持续改进直到达到质量要求。

In [ ]:
def route(state: State):
    return "Accept" if state["grade"] == "pass" else "Revise"

# 构建 LangGraph
定义基于 LangGraph 的工作流，将生成器和评估器分别加入为节点。有向边规定先生成再评估；条件边根据路由逻辑，在通过时终止循环，在失败时返回生成器。这种结构支持自我修正，每轮迭代都可通过 Phoenix 追踪。

In [ ]:
builder = StateGraph(State)

builder.add_node("code_generator", code_generator)
builder.add_node("code_evaluator", code_evaluator)

builder.add_edge(START, "code_generator")
builder.add_edge("code_generator", "code_evaluator")
builder.add_conditional_edges(
    "code_evaluator",
    route,
    {
        "Accept": END,
        "Revise": "code_generator",
    },
)

workflow = builder.compile()

# 使用示例

In [ ]:
problem = """
Write code for a complicated website with javascript features.
"""

result_state = workflow.invoke({"problem_spec": problem})
print("===== 最终代码 =====\n")
print(result_state["code"])

# 记得在 Phoenix 中查看追踪！